# SN-06 — Routage des PM non résolues vers leur autorité d'enregistrement

Reprend les personnes morales dont la phase 3 se termine en `DOUTEUX` ou
`REJETE` et les oriente vers l'autorité compétente.

La catégorie d'établissement est un attribut d'EGE, pas de PM. L'autorité d'une
personne morale se déduit donc de ses établissements, selon deux règles.

**Un seul EGE rattaché** : la PM hérite intégralement du résultat de cet
établissement — code, statut et motif — qu'il ait été résolu par sa catégorie ou
par l'autorité déclarée dans FINESS+.

**Plusieurs EGE** : on cherche la famille d'autorité majoritaire parmi les
domaines des catégories des fils. Sanitaire et médico-social votent ARS, social
et formation votent solidarités. La majorité doit être stricte : sur deux fils
de familles différentes, aucune ne l'emporte et la PM part en arbitrage. Le
département retenu reste celui de la PM.

**Rattrapage** : comme pour les EGE, les PM encore `A_ARBITRER` ou
`NON_DETERMINEE` sont reprises avec l'autorité déclarée dans FINESS+, lue sur le
dernier gestionnaire ayant mis à jour la structure.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from src import autorites as aut, routage as rt
from src.connexion   import get_finess_connection
from src.chargement  import resoudre_tables, charger_contacts
from src.display     import afficher_tableau
from config.settings import (
    SN_PHASE3, FINESS_EGE_CLEAN, REFERENTIEL_CATEGORIES, ROUTAGE_DIR,
)

CLE = rt.CLE_PM

## 1. Référentiel des catégories

In [2]:
referentiel = aut.charger_referentiel(REFERENTIEL_CATEGORIES)
print(f"{len(referentiel):,} categories")
print(referentiel["famille"].value_counts().to_string())

181 categories
famille
ARS            135
SOLIDARITES     46


## 2. Autorité déclarée dans FINESS+

In [3]:
conn = get_finess_connection()
TABLES = resoudre_tables(conn, {
    "pm": "PM_SMSSE", "ege": "EGE",
    "metadonnee": "Metadonnee", "gestionnaire": "Gestionnaire",
    "telecom": "Telecom",
    "pm_contact": "PmSmsseContact",
})
ae_par_meta = rt.charger_ae_declaree(conn, TABLES)

print("\nCoordonnees :")
contacts = charger_contacts(conn, TABLES, "pm")
conn.close()

display(ae_par_meta[ae_par_meta["ae_declaree_finess"] != ""]["ae_declaree_finess"]
        .value_counts().head(20).to_frame("nb_metadonnees"))

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   metadonnee   -> T_FAC_Metadonnee
   gestionnaire -> T_FAC_Gestionnaire
   telecom      -> T_FAC_Telecom
   pm_contact   -> T_FAC_PmSmsseContact
   158,254 metadonnees rattachees a un gestionnaire, 158,251 avec une autorite exploitable

Coordonnees :
   79,723 PM avec au moins une coordonnee (34,568 mail, 76,511 telephone)


,nb_metadonnees
ae_declaree_finess,
ANS,58716
ARS-11,12127
ARS-75,9758
ARS-84,9712
ARS-76,8117
ARS-53,6156
ARS-93,6137
ARS-44,5887
ARS-32,5042


## 3. Structures à router

In [4]:
df = rt.charger_phase3(SN_PHASE3, col_cle=CLE)
print(f"{df[CLE].nunique():,} PM, {len(df):,} lignes (top 3 conserve)")
print(df.drop_duplicates(CLE)["statut_structure"].value_counts().to_string())

2,364 PM, 7,092 lignes (top 3 conserve)
statut_structure
DOUTEUX    2334
REJETE       30


## 4. Établissements rattachés

Lus dans le parquet du chargement FINESS, sur `NB_PmSmsseId`. Ce parquet est
déjà restreint aux structures actives et garantit que la remontée porte sur le
même millésime que les phases amont.

In [5]:
ege_fils = rt.charger_ege_fils(FINESS_EGE_CLEAN, df["NB_PmSmsseId"].dropna().unique())
print(f"{len(ege_fils):,} EGE rattaches a {ege_fils['NB_PmSmsseId'].nunique():,} PM")

repartition = ege_fils.groupby("NB_PmSmsseId").size().value_counts().sort_index()
print()
print("Nombre d'EGE par PM :")
print(repartition.head(10).to_string())

2,086 EGE rattaches a 1,435 PM

Nombre d'EGE par PM :
1     1253
2       94
3       29
4       20
5        3
6        5
7        1
8        3
9        7
10       2


## 5. Routage

In [6]:
resultats = rt.router_sirenisation(df, referentiel, ege_fils, ae_par_meta)

rang1 = resultats[pd.to_numeric(resultats["rang"], errors="coerce") == 1]
print(rang1["autorite_statut"].value_counts().to_string())
print()
print("Origine de la resolution :")
print(rang1[rang1["autorite_statut"] == aut.RESOLU]["autorite_source"]
      .value_counts().to_string())
print()
print("Motifs des cas non resolus :")
print(rang1[rang1["autorite_statut"] != aut.RESOLU]["autorite_motif"]
      .value_counts().to_string())

   34 structures resolues par l'autorite declaree
   FK_MetadonneeId absente, rattrapage ignore
autorite_statut
RESOLU            1245
NON_DETERMINEE    1105
A_ARBITRER          14

Origine de la resolution :
autorite_source
EGE_UNIQUE    1245

Motifs des cas non resolus :
autorite_motif
aucun_ege_rattache               929
departement_pm_hors_perimetre    176
categorie_multi_autorites          8
domaine_fils_partage               6


In [7]:
partages = rang1[rang1["autorite_motif"] == "domaine_fils_partage"]
print(f"{len(partages):,} PM sans majorite de domaine")
if len(partages):
    afficher_tableau(
        partages[[CLE, "nb_ege_fils", "familles_fils", "autorite_candidats"]],
        "PM a arbitrer : domaines partages entre les fils", max_lignes=15)

6 PM sans majorite de domaine


TX_NumFinessPm,nb_ege_fils,familles_fils,autorite_candidats
040003170,2,SOLIDARITES:1|ARS:1,
300784170,2,SOLIDARITES:1|ARS:1,
460784549,2,SOLIDARITES:1|ARS:1,
600001937,2,SOLIDARITES:1|ARS:1,
610001000,2,ARS:1|SOLIDARITES:1,
750828717,20,SOLIDARITES:10|ARS:10,


In [8]:
afficher_tableau(
    rang1.groupby(["autorite_code", "libelle_region"])[CLE].nunique()
         .reset_index(name="nb_pm").sort_values("nb_pm", ascending=False),
    "Repartition par autorite", max_lignes=25)

autorite_code,libelle_region,nb_pm
,,1119
ARS-76,Occitanie,155
ARS-75,Nouvelle-Aquitaine,143
ARS-84,Auvergne-Rhône-Alpes,132
ARS-93,Provence-Alpes-Côte d'Azur,110
ARS-11,Île-de-France,97
ARS-32,Hauts-de-France,85
ARS-27,Bourgogne-Franche-Comté,66
ARS-53,Bretagne,63
ARS-52,Pays de la Loire,53


## Coordonnées de contact

Lues directement dans FINESS+ : la liaison contact, puis la table Telecom
pivotée par canal. Les coordonnées de tous les contacts d'une structure sont
regroupées, ceux-ci étant souvent spécialisés — l'un porte le téléphone, l'autre
le mail.


In [9]:
resultats = rt.joindre_contacts(resultats, contacts, "NB_PmSmsseId")


   TEL_MAIL         2,316 renseignes
   TEL_TELEPHONE    5,439 renseignes
   TEL_TELECOPIE    2,520 renseignes


## 7. Export

In [10]:
COLONNES_EXPORT = [
    "autorite_code", "libelle_autorite", "libelle_region", "departement",
    "autorite_statut", "autorite_source", "autorite_candidats", "autorite_motif",
    "ae_declaree_finess", "origine_categorie", "nb_ege_fils", "familles_fils",
    "TX_NumFinessPm", "NB_PmSmsseId", "TX_DenominationPm",
    "adresse_complete_pm", "TEL_MAIL", "TEL_TELEPHONE",
    "TX_Siren", "siren_ref_app", "nom_ul_retenu",
    "adresse_siege_complete_ul", "activitePrincipaleUniteLegale",
    "score_nom", "score_adresse", "score_global", "score_global_approfondi",
    "statut_candidat", "statut_structure", "rang",
]
COLONNES_EXPORT = [c for c in COLONNES_EXPORT if c in resultats.columns]

recap = rt.exporter(resultats, ROUTAGE_DIR / "sirenisation", "SN_P3",
                    col_cle=CLE, colonnes=COLONNES_EXPORT)
afficher_tableau(recap, "Classeurs ecrits", max_lignes=30)

attendu = resultats[CLE].nunique()
obtenu = int(recap["nb_structures"].sum())
assert attendu == obtenu, f"perte au decoupage : {attendu} != {obtenu}"
print(f"\n{obtenu:,} PM reparties sur {len(recap)} classeurs")

autorite,region,fichier,nb_structures,nb_lignes,nb_departements
NON_DETERMINEE,,SN_P3_NON_DETERMINEE.xlsx,1105,3315,99
ARS-76,Occitanie,SN_P3_ARS-76_Occitanie.xlsx,155,465,14
ARS-75,Nouvelle-Aquitaine,SN_P3_ARS-75_Nouvelle-Aquitaine.xlsx,143,429,12
ARS-84,Auvergne-Rhône-Alpes,SN_P3_ARS-84_Auvergne-Rhone-Alpes.xlsx,132,396,12
ARS-93,Provence-Alpes-Côte d'Azur,SN_P3_ARS-93_Provence-Alpes-Cote-d-Azur.xlsx,110,330,7
ARS-11,Île-de-France,SN_P3_ARS-11_Ile-de-France.xlsx,97,291,10
ARS-32,Hauts-de-France,SN_P3_ARS-32_Hauts-de-France.xlsx,85,255,5
ARS-27,Bourgogne-Franche-Comté,SN_P3_ARS-27_Bourgogne-Franche-Comte.xlsx,66,198,8
ARS-53,Bretagne,SN_P3_ARS-53_Bretagne.xlsx,63,189,4
ARS-52,Pays de la Loire,SN_P3_ARS-52_Pays-de-la-Loire.xlsx,53,159,5



2,364 PM reparties sur 37 classeurs
